# Obtain Stage — Facial Expression Recognition (Smile / Neutral / Sad)

Self-contained **Obtain** pipeline (OSEMN) for **still-image datasets**: public datasets
(FER2013, CK+, AffectNet, KDEF, RAF-DB, ...) and team-collected photos alike.
It replaces the old webcam/video Obtain loop from `AssistiveDomotical` with a
folder-of-images pipeline built on **MediaPipe FaceMesh** (`mediapipe==0.10.21`, solutions API).

**Inputs** — a directory tree of images (default `./data/images`, any layout, see folder
convention below). No webcam, no video required.

**Outputs** (written to `./outputs`, all overwritten each run — idempotent):

| File | Content |
|---|---|
| `manifest.csv` | one row per image found: provenance + quality metrics + face-detection outcome |
| `rejected_images.csv` | images gated at ingestion with human-readable reasons |
| `geometric_features.csv` | one row per accepted image: scalar IOD-normalized geometry (Approach A) |
| `landmark_features.csv` | one row per accepted image: normalized landmark coordinates (Approach B) |
| `landmark_index.csv` | maps each `x_i / y_i [/ z_i]` column to its FaceMesh landmark index |
| `crops/raw/`, `crops/aligned/` | optional face crops for downstream CNN experiments |

**Run order:** top to bottom. The only cell to edit is the demo cell at the end.


## Expected folder convention (for subject / source / label parsing)

Parsing is *best-effort* and driven by `parse_source` / `parse_subject_id` / `parse_label`:

```
data/images/                      ← image_dir (any name; passed as a parameter)
├── kdef/                         ← source  = first folder under image_dir
│   ├── smile/                    ← label   = any folder named smile/neutral/sad (aliases OK)
│   │   └── kevin_01.jpg          ← subject = parent folder when it is not a label folder
│   ├── neutral/…
│   └── sad/…
└── team_photos/                  ← source
    ├── felipe/                   ← subject = folder (no label folder → label from filename)
    │   ├── sess1_smile_01.jpg    ← label   = "smile" token in the filename
    │   └── sess1_neutral_01.jpg
    └── ana/
        └── img_01.jpg            ← no label anywhere → label = None (annotate later, in Scrub)
```

**Label assumption (documented, not hidden):** a label is recognized only when a folder
component, or a filename token (split on spaces / `_` / `-` / `.`), case-insensitively
matches one of the three classes or their aliases (`happy`→smile, `neutro`→neutral,
`triste`→sad, …). Anything else → `label = None`. This notebook **never predicts labels** —
it only harvests labels that already exist in your folder/file naming.


## Design decisions (stated up front)

| # | Decision | Why |
|---|---|---|
| D1 | **2D landmarks by default** in the coordinate table; `include_z=True` opt-in | FaceMesh's monocular `z` is noisy and on a different scale than x/y; 2D keeps the CSV lean for classical ML. The flag lets you experiment later. |
| D2 | **Wide landmark table** (`x_0..y_477`) + companion `landmark_index.csv` | Wide matches the old repo's `model_pontos.py` consumption pattern (sklearn wants one row per sample); the index file keeps the mapping auditable. |
| D3 | **All distances normalized by inter-ocular distance (IOD)**; `ear`/`mar`/`mouth_curvature` are already dimensionless | Absolute pixel coordinates (old repo) do not transfer across image sizes/camera distances. IOD is stable across expressions (eyes don't move), unlike face width which changes with smiles. Raw `iod_px` is kept as a column for scale context. |
| D4 | FaceMesh configured with `max_num_faces=5` + **largest-face-by-bbox selection** | Borrowed from `observableDlib.verificandoMaiorFace()`. With `max_num_faces=1` FaceMesh returns one arbitrary face, so the heuristic needs candidates to choose from. |
| D5 | **6-point EAR** per eye (generalizes the old 4-point ratio from landmarks 159/145/33/133) | `padroes/equacoes.py::razaoDistOlhos` *is* the classic 6-point formula `(‖p2−p6‖+‖p3−p5‖)/(2‖p1−p4‖)`; the old MediaPipe code just fed it fewer points. We restore the full 6-point ring per eye using the FaceMesh↔dlib correspondence. |
| D6 | Rejection thresholds: Laplacian variance < 100, mean gray < 40 or > 220, short side < 64 px | 100 is the common blur heuristic; 40/220 catch under/over-exposure; 64 px is roughly where FaceMesh reliability collapses. All are CONFIG entries — tune per dataset. |
| D7 | Alignment = similarity transform mapping eye centers to fixed canonical positions | Makes crops size/rotation-invariant for future CNN baselines. Geometric/landmark tables don't need it (they are already IOD-normalized). |
| D8 | **Idempotent**: every CSV is overwritten; `crops/` is wiped (only when `save_crops=True`) before regeneration | Fixes the old repo's shared-class-attribute accumulation bug (`criandoTabelas.py`): re-running must never duplicate rows. |
| D9 | `left`/`right` follow MediaPipe's **anatomical** convention | The subject's right eye (indices 33/133/159/145…) appears on the *image's left* in non-mirrored photos. Stated once here to avoid sign confusion in downstream analysis. |
| D10 | Rejected images are **excluded** from feature tables but **present** in `manifest.csv` and `rejected_images.csv` | "Never silently drop": every image lands somewhere. Feature tables stay clean for Model stage. |


In [ ]:
# =============================================================================
# Imports + configuration (all thresholds and landmark indices are named here)
# =============================================================================
from __future__ import annotations

import logging
import math
import re
import shutil
from contextlib import contextmanager
from pathlib import Path
from typing import Any, Dict, Generator, Iterator, List, Optional, Sequence, Tuple, Union

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd

try:  # optional dependency — the pipeline works without it
    from tqdm.auto import tqdm
except ImportError:  # pragma: no cover
    def tqdm(iterable, **kwargs):  # type: ignore[misc]
        return iterable

logging.basicConfig(level=logging.INFO, format="%(levelname)-7s | %(message)s")
LOGGER = logging.getLogger("obtain_fer")

PathLike = Union[str, Path]

# ---------------------------------------------------------------------------
# Global configuration — every knob lives here, nothing is hard-coded elsewhere
# ---------------------------------------------------------------------------
CONFIG: Dict[str, Any] = {
    # --- I/O -----------------------------------------------------------------
    "image_dir": "./data/images",           # default input; always overridable
    "output_dir": "./outputs",              # default output; always overridable
    "extensions": (".jpg", ".jpeg", ".png", ".bmp"),
    # --- FaceMesh (mediapipe==0.10.21, solutions API) ------------------------
    "static_image_mode": True,              # still images, not video tracking
    "max_num_faces": 5,                     # >1 so the largest-face heuristic can choose (D4)
    "refine_landmarks": True,               # adds 10 iris landmarks (478 total vs 468)
    "min_detection_confidence": 0.5,
    "min_tracking_confidence": 0.5,         # unused in static mode; kept for API parity
    # --- ingestion-time validation (D6) --------------------------------------
    "blur_threshold": 100.0,                # min Laplacian variance
    "brightness_low": 40.0,                 # mean gray below => too dark
    "brightness_high": 220.0,               # mean gray above => too bright
    "min_resolution": 64,                   # min(width, height) in px
    # --- crop / alignment ------------------------------------------------------
    "crop_margin": 0.20,                    # fraction of bbox size padded around landmarks
    "align_output_size": (224, 224),        # canonical crop size (CNN-friendly)
    "save_crops": False,                    # opt-in: writing crops is slow on big datasets
    # --- landmark coordinate table -------------------------------------------
    "include_z": False,                     # D1: 2D by default
    "landmark_indices": None,               # None => all FaceMesh landmarks
}

# Three-class problem; aliases let teams keep their existing folder/file names.
CLASS_NAMES: Tuple[str, ...] = ("smile", "neutral", "sad")
CLASS_ALIASES: Dict[str, str] = {
    "smile": "smile", "smiling": "smile", "happy": "smile", "sorriso": "smile", "sorrindo": "smile",
    "neutral": "neutral", "neutro": "neutral", "neutra": "neutral",
    "sad": "sad", "sorrow": "sad", "triste": "sad",
}
# Filename tokens that should NOT be mistaken for a subject id.
GENERIC_SUBJECT_TOKENS = {"img", "image", "images", "frame", "photo", "foto", "sample", "test", "face", "crop"}

# ---------------------------------------------------------------------------
# FaceMesh landmark index constants — no magic numbers below this cell.
# Naming is ANATOMICAL (D9): the subject's right eye (33-group) appears
# image-left in non-mirrored photos.
# ---------------------------------------------------------------------------
# 6-point Eye Aspect Ratio rings. Correspondence with the dlib 68-point model
# (which `padroes/equacoes.py::razaoDistOlhos` was written for):
#   right eye: dlib 36,37,38,39,40,41  ->  FaceMesh 33,160,158,133,153,144
#   left  eye: dlib 42,43,44,45,46,47  ->  FaceMesh 362,385,387,263,373,380
EYE_RIGHT: Dict[str, int] = {"p1": 33, "p2": 160, "p3": 158, "p4": 133, "p5": 153, "p6": 144}
EYE_LEFT: Dict[str, int] = {"p1": 362, "p2": 385, "p3": 387, "p4": 263, "p5": 373, "p6": 380}
EYE_RIGHT_IDXS: Tuple[int, ...] = tuple(EYE_RIGHT.values())
EYE_LEFT_IDXS: Tuple[int, ...] = tuple(EYE_LEFT.values())
ALL_EYE_IDXS: Tuple[int, ...] = EYE_RIGHT_IDXS + EYE_LEFT_IDXS

MOUTH_CORNER_RIGHT = 61    # image-left corner
MOUTH_CORNER_LEFT = 291    # image-right corner
LIP_UPPER_INNER = 13
LIP_LOWER_INNER = 14
LIP_UPPER_OUTER = 0
LIP_LOWER_OUTER = 17

NOSE_TIP = 1
CHIN_BOTTOM = 152
FOREHEAD_TOP = 10
FACE_SIDE_RIGHT = 234      # image-left face-oval boundary
FACE_SIDE_LEFT = 454       # image-right face-oval boundary

BROW_RIGHT = 105           # mid upper-brow point, subject's right
BROW_LEFT = 334            # mid upper-brow point, subject's left

IRIS_RIGHT_CENTER = 468    # present only when refine_landmarks=True
IRIS_LEFT_CENTER = 473

# Every landmark index referenced by the geometric features (bounds-checking).
MAX_USED_INDEX = max(*EYE_RIGHT_IDXS, *EYE_LEFT_IDXS, MOUTH_CORNER_RIGHT, MOUTH_CORNER_LEFT,
                     LIP_UPPER_INNER, LIP_LOWER_INNER, LIP_UPPER_OUTER, LIP_LOWER_OUTER,
                     NOSE_TIP, CHIN_BOTTOM, FOREHEAD_TOP, FACE_SIDE_RIGHT, FACE_SIDE_LEFT,
                     BROW_RIGHT, BROW_LEFT)

# Canonical column order for the geometric table (schema stability).
GEOMETRIC_FEATURE_ORDER: Tuple[str, ...] = (
    "ear_left", "ear_right", "mar", "mouth_width", "mouth_height",
    "brow_eye_distance_left", "brow_eye_distance_right", "mouth_curvature",
    "nose_to_mouth_corner_left", "nose_to_mouth_corner_right",
    "face_width", "face_height",
)
GEOMETRIC_META_COLUMNS: Tuple[str, ...] = (
    "image_path", "source", "subject_id", "label", "face_found", "n_faces",
)
GEOMETRIC_COLUMNS: Tuple[str, ...] = GEOMETRIC_META_COLUMNS + ("iod_px",) + GEOMETRIC_FEATURE_ORDER + ("error",)

MANIFEST_COLUMNS: Tuple[str, ...] = (
    "image_path", "source", "subject_id", "label",
    "width", "height", "face_found", "n_landmarks", "blur_score", "brightness", "rejected_reason",
)
REJECTED_COLUMNS: Tuple[str, ...] = (
    "image_path", "source", "subject_id", "label",
    "width", "height", "blur_score", "brightness", "rejected_reason",
)
LANDMARK_META_COLUMNS: Tuple[str, ...] = (
    "image_path", "source", "subject_id", "label", "face_found", "n_landmarks", "error",
)


## 1. Image reading

Three layers:
* `read_image` — one file → BGR array (or `None`). Reads through `np.fromfile` +
  `cv2.imdecode` instead of `cv2.imread` because `imread` silently fails on
  non-ASCII paths on Windows — a real risk with team-collected photos.
* `list_images` — recursive, sorted, skips hidden files/dirs (`.ipynb_checkpoints`, `.DS_Store`…).
  **Fails loudly** (`FileNotFoundError`) if the root does not exist.
* `load_images` — generator yielding `(path, image)`. Unreadable files are **logged and
  yielded as `(path, None)`** so callers can audit them — never silently dropped (D10).


In [ ]:
def read_image(path: PathLike, to_rgb: bool = False) -> Optional[np.ndarray]:
    """Load a single image as a BGR uint8 array (optionally RGB).

    Parameters
    ----------
    path:
        File path to an image.
    to_rgb:
        If True, convert BGR→RGB. MediaPipe FaceMesh expects RGB; OpenCV
        drawing/imshow expects BGR, so conversion is left to the caller.

    Returns
    -------
    np.ndarray (H, W, 3) or None when the file cannot be read or decodes empty.
    """
    try:
        data = np.fromfile(str(path), dtype=np.uint8)  # unicode-safe on Windows
    except (OSError, ValueError) as exc:
        LOGGER.debug("read_image: os/decode error for %s (%s)", path, exc)
        return None
    image = cv2.imdecode(data, cv2.IMREAD_COLOR)  # always BGR
    if image is None or image.size == 0:
        LOGGER.debug("read_image: empty/undecodable image: %s", path)
        return None
    if to_rgb:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return image


def list_images(
    image_dir: PathLike,
    extensions: Sequence[str] = CONFIG["extensions"],
) -> List[Path]:
    """Return a sorted list of image file paths under ``image_dir`` (recursive).

    Skips non-image extensions, hidden files and anything inside a hidden
    directory (e.g. ``.ipynb_checkpoints``). Raises ``FileNotFoundError`` when
    ``image_dir`` does not exist — missing input must fail loudly, not quietly
    produce empty tables.
    """
    root = Path(image_dir)
    if not root.is_dir():
        raise FileNotFoundError(f"Input image directory not found: {root.resolve()}")
    wanted = {ext.lower() for ext in extensions}
    paths: List[Path] = []
    for candidate in sorted(root.rglob("*")):
        if not candidate.is_file():
            continue
        rel_parts = candidate.relative_to(root).parts
        if any(part.startswith(".") for part in rel_parts):  # hidden dir/file
            continue
        if candidate.suffix.lower() in wanted:
            paths.append(candidate)
    return paths


def load_images(
    image_dir: PathLike,
    extensions: Sequence[str] = CONFIG["extensions"],
) -> Generator[Tuple[Path, Optional[np.ndarray]], None, None]:
    """Yield ``(path, image_bgr)`` for every readable image under ``image_dir``.

    Unreadable files yield ``(path, None)`` after a logged warning — the caller
    decides what to do, but nothing is ever dropped silently (D10).
    """
    for path in list_images(image_dir, extensions):
        image = read_image(path)
        if image is None:
            LOGGER.warning("Unreadable image, skipping processing: %s", path)
        yield path, image


## 2. MediaPipe landmark extraction

Faithful to `observables/observableMediaPipe.py` where the old code was right
(FaceMesh config, RGB conversion, normalized→pixel scaling) and generalized
where it was not:

* **Static mode** (`static_image_mode=True`) — the old code processed video and
  relied on tracking between frames; for still images every frame must be
  detected independently.
* **Largest-face selection (D4)** — borrowed from `observableDlib.verificandoMaiorFace()`,
  but computed from normalized bounding-box area instead of dlib landmark extents.
  This is why the pipeline runs FaceMesh with `max_num_faces=5` even though we keep one face.
* **Manual pixel scaling** `(round(x·W), round(y·H))` replaces the old call to the
  private `mp.solutions.drawing_utils._normalized_to_pixel_coordinates` — same math,
  no private-API dependency.
* The result is a plain **dict**, not attributes strewn across `self` like the old
  `ObservableMediaPipe` — that stateful style is what made the old collector
  accumulate rows across videos.


In [ ]:
def init_face_mesh(
    static_image_mode: bool = True,
    max_num_faces: int = 1,
    refine_landmarks: bool = True,
    min_detection_confidence: float = 0.5,
    min_tracking_confidence: float = 0.5,
) -> "mp.solutions.face_mesh.FaceMesh":
    """Create a configured MediaPipe FaceMesh (mediapipe==0.10.21 solutions API).

    Defaults match the old repo's `confFaceDetection()` except `static_image_mode`,
    which must be True for still images (tracking across unrelated photos is meaningless).

    Note (D4): the pipeline calls this with `max_num_faces=CONFIG["max_num_faces"]`
    (5) so that `extract_landmarks` can pick the largest face by bounding-box area.
    """
    return mp.solutions.face_mesh.FaceMesh(
        static_image_mode=static_image_mode,
        max_num_faces=max_num_faces,
        refine_landmarks=refine_landmarks,
        min_detection_confidence=min_detection_confidence,
        min_tracking_confidence=min_tracking_confidence,
    )


@contextmanager
def face_mesh_session(face_mesh: Optional["mp.solutions.face_mesh.FaceMesh"] = None) -> Iterator:
    """Context manager yielding a FaceMesh instance, closing it only if we created it.

    Lets callers reuse one FaceMesh across builders (FaceMesh setup is expensive)
    while guaranteeing cleanup for standalone use.
    """
    owns = face_mesh is None
    instance = face_mesh or init_face_mesh(
        static_image_mode=CONFIG["static_image_mode"],
        max_num_faces=CONFIG["max_num_faces"],
        refine_landmarks=CONFIG["refine_landmarks"],
        min_detection_confidence=CONFIG["min_detection_confidence"],
        min_tracking_confidence=CONFIG["min_tracking_confidence"],
    )
    try:
        yield instance
    finally:
        if owns:
            instance.close()


def _largest_face(face_landmarks_list: Sequence, image_shape: Tuple[int, ...]):
    """Return the FaceMesh landmark object with the largest bounding-box area.

    Generalizes `observableDlib.verificandoMaiorFace()` (which compared dlib
    landmark extents) to normalized FaceMesh coordinates; normalized area is
    comparable because all candidates share the same image.
    """
    best, best_area = None, -1.0
    for face_landmarks in face_landmarks_list:
        xs = [lm.x for lm in face_landmarks.landmark]
        ys = [lm.y for lm in face_landmarks.landmark]
        area = (max(xs) - min(xs)) * (max(ys) - min(ys))
        if area > best_area:
            best, best_area = face_landmarks, area
    return best


def extract_landmarks(
    image_rgb: np.ndarray,
    face_mesh: "mp.solutions.face_mesh.FaceMesh",
) -> Dict[str, Any]:
    """Run FaceMesh on one RGB image and return a structured result.

    Returns
    -------
    dict with keys:
        face_found      : bool
        landmarks_norm  : list[(x, y, z)] normalized to [0,1] (x/y; z is relative depth), or None
        landmarks_px    : list[(x, y)] integer pixel coordinates, or None
        n_faces         : int — number of faces detected before largest-face selection
        error           : None or a short human-readable reason
    """
    result: Dict[str, Any] = {
        "face_found": False,
        "landmarks_norm": None,
        "landmarks_px": None,
        "n_faces": 0,
        "error": None,
    }
    if image_rgb is None or getattr(image_rgb, "size", 0) == 0:
        result["error"] = "empty image"
        return result
    try:
        outputs = face_mesh.process(image_rgb)
    except Exception as exc:  # FaceMesh can raise on pathological inputs; log, don't crash
        result["error"] = f"facemesh failure: {exc}"
        LOGGER.warning("FaceMesh failed on an image: %s", exc)
        return result

    multi = list(outputs.multi_face_landmarks or [])
    result["n_faces"] = len(multi)
    if not multi:
        result["error"] = "no face detected"
        return result

    chosen = _largest_face(multi, image_rgb.shape)  # D4
    height, width = image_rgb.shape[:2]
    landmarks = chosen.landmark
    # Manual normalized→pixel conversion (replaces the private
    # drawing_utils._normalized_to_pixel_coordinates used by the old repo).
    result["landmarks_norm"] = [(lm.x, lm.y, lm.z) for lm in landmarks]
    result["landmarks_px"] = [
        (int(round(lm.x * width)), int(round(lm.y * height))) for lm in landmarks
    ]
    result["face_found"] = True
    return result


## 3. Geometric-approach table (Approach A — scalar measurements)

**Normalization (D3).** Every distance-type feature is divided by the
**inter-ocular distance (IOD)** — the distance between the two eye centers
(each center = mean of that eye's six EAR landmarks). Rationale:

* absolute pixel values (old repo) are not comparable across image sizes / camera distances;
* IOD is nearly invariant to expression (the eye's bony orbit doesn't move when you smile),
  unlike `face_width` or mouth-based scales;
* `ear`, `mar` and `mouth_curvature` are already ratios and need no normalization.

**EAR generalization (D5).** The old MediaPipe pipeline computed a 2-vertical/1-horizontal
ratio from landmarks 159/145/33/133. We use the full 6-point ring per eye — which is exactly
the formula in `padroes/equacoes.py::razaoDistOlhos`, fed with the FaceMesh↔dlib
correspondence (dlib 36–41 → FaceMesh 33,160,158,133,153,144; dlib 42–47 → 362,385,387,263,373,380).

**Sign convention.** In image coordinates y grows **downward**.
`mouth_curvature = (mean corner y − inner-lip-center y) / mouth_width`,
so **negative ⇒ corners above lip center ⇒ smile-like upturn**; positive ⇒ downturn (sad).


In [ ]:
def _dist(p: Sequence[float], q: Sequence[float]) -> float:
    """Euclidean distance between two 2D points."""
    return float(math.hypot(float(p[0]) - float(q[0]), float(p[1]) - float(q[1])))


def _eye_aspect_ratio(points: Sequence[Sequence[float]], eye: Dict[str, int]) -> float:
    """6-point Eye Aspect Ratio — same formula as `padroes/equacoes.py::razaoDistOlhos`:
    (‖p2−p6‖ + ‖p3−p5‖) / (2·‖p1−p4‖)  with p1/p4 the outer/inner eye corners.
    """
    p1, p2, p3, p4, p5, p6 = (points[eye[k]] for k in ("p1", "p2", "p3", "p4", "p5", "p6"))
    vertical = _dist(p2, p6) + _dist(p3, p5)
    horizontal = _dist(p1, p4)
    if horizontal <= 0:
        return float("nan")
    return vertical / (2.0 * horizontal)


def compute_geometric_features(landmarks_px: Sequence[Sequence[float]]) -> Dict[str, float]:
    """Compute IOD-normalized scalar geometry for one face.

    Parameters
    ----------
    landmarks_px:
        Pixel-coordinate landmarks as an (N, 2)-or-more sequence; FaceMesh indices
        are used to select points, so N must be the full FaceMesh set (468/478).

    Returns
    -------
    dict of scalar features (see GEOMETRIC_FEATURE_ORDER for the canonical order):
        ear_left, ear_right          6-point EAR per eye (dimensionless)
        mar                          mouth aspect ratio = mouth_height / mouth_width
        mouth_width, mouth_height    IOD-normalized
        brow_eye_distance_left/right brow point (334/105) to eye center, IOD-normalized
        mouth_curvature              (mean corner y − inner-lip-center y) / mouth_width;
                                     negative = smile-like upturn (see sign note above)
        nose_to_mouth_corner_left/right  nose tip (1) to corners 291/61, IOD-normalized
        face_width, face_height      face-oval extents (234–454, 10–152), IOD-normalized
        iod_px                       raw inter-ocular distance in px (scale context)
    """
    pts = [(float(p[0]), float(p[1])) for p in landmarks_px]
    if len(pts) <= MAX_USED_INDEX:
        raise ValueError(
            f"expected >= {MAX_USED_INDEX + 1} FaceMesh landmarks, got {len(pts)}"
        )

    eye_right_center = np.mean([pts[i] for i in EYE_RIGHT_IDXS], axis=0)
    eye_left_center = np.mean([pts[i] for i in EYE_LEFT_IDXS], axis=0)
    iod = float(np.linalg.norm(eye_left_center - eye_right_center))

    def by_iod(value: float) -> float:
        return value / iod if iod > 0 else float("nan")

    corner_right, corner_left = pts[MOUTH_CORNER_RIGHT], pts[MOUTH_CORNER_LEFT]
    mouth_width = _dist(corner_right, corner_left)
    mouth_height = _dist(pts[LIP_UPPER_INNER], pts[LIP_LOWER_INNER])

    # Smile upturn: corners sit ABOVE the inner-lip center (smaller y) -> negative.
    lip_center_y = (pts[LIP_UPPER_INNER][1] + pts[LIP_LOWER_INNER][1]) / 2.0
    corners_mean_y = (corner_right[1] + corner_left[1]) / 2.0
    mouth_curvature = (
        (corners_mean_y - lip_center_y) / mouth_width if mouth_width > 0 else float("nan")
    )

    return {
        "iod_px": iod,
        "ear_left": _eye_aspect_ratio(pts, EYE_LEFT),
        "ear_right": _eye_aspect_ratio(pts, EYE_RIGHT),
        "mar": mouth_height / mouth_width if mouth_width > 0 else float("nan"),
        "mouth_width": by_iod(mouth_width),
        "mouth_height": by_iod(mouth_height),
        "brow_eye_distance_left": by_iod(_dist(pts[BROW_LEFT], eye_left_center)),
        "brow_eye_distance_right": by_iod(_dist(pts[BROW_RIGHT], eye_right_center)),
        "mouth_curvature": mouth_curvature,
        "nose_to_mouth_corner_left": by_iod(_dist(pts[NOSE_TIP], corner_left)),
        "nose_to_mouth_corner_right": by_iod(_dist(pts[NOSE_TIP], corner_right)),
        "face_width": by_iod(_dist(pts[FACE_SIDE_RIGHT], pts[FACE_SIDE_LEFT])),
        "face_height": by_iod(_dist(pts[FOREHEAD_TOP], pts[CHIN_BOTTOM])),
    }


## 4. Ratio / landmark-coordinate approach table (Approach B — point distribution)

Instead of hand-picked ratios, this representation is the **normalized landmark
coordinates themselves** — the generalization of the old repo's `pontos_olho`
pixel tuples that `model_pontos.py` flattened into `x_j/y_j` columns.

**Choices (stated before the code, as required):**

* **2D vs 3D (D1):** default **2D** (`x_i, y_i`), `include_z=True` adds `z_i`.
  Reason: FaceMesh monocular `z` is noisy, centered arbitrarily, and doubles the
  column count; start lean, keep the door open. When enabled, `z` is re-centered on
  its mean and divided by the *same* IOD scale factor as x/y (≈ consistent units
  under the weak-perspective assumption).
* **Centering / scaling:** coordinates are centered on the **mean of all 12 EAR eye
  landmarks** (≈ mid-eyes point, robust to expression) and divided by **IOD** —
  the same scale reference as Approach A, so both tables share units.
* **Wide vs long (D2):** **wide** (one row per image, `x_0..y_477` columns) because
  sklearn consumes it directly — the old `model_pontos.py` had to `ast.literal_eval`
  stringified tuples out of CSVs; wide + a companion `landmark_index.csv` avoids
  that fragility while keeping the mapping auditable.

⚠ Pass **pixel** coordinates (the pipeline does). Feeding *normalized* coords would
divide x and y by different factors (image width vs height) when the image is not square,
distorting geometry.


In [ ]:
def normalize_landmarks(
    landmarks: Sequence[Sequence[float]],
    return_z: bool = False,
    z_values: Optional[Sequence[float]] = None,
) -> Optional[np.ndarray]:
    """Center landmarks on the mid-eyes point and scale by inter-ocular distance.

    Parameters
    ----------
    landmarks:
        (N, 2) or (N, 3) array-like of ONE face's landmarks. Pixel coordinates are
        strongly recommended (see aspect-ratio warning above); normalized coords
        also work but bake the image aspect ratio into the result.
    return_z:
        If True, append a re-centered, IOD-scaled z column. The z values are taken
        from the input's third column when present, otherwise from ``z_values``.
        z must be expressed in the SAME pixel scale as x/y: FaceMesh's native z
        is roughly x-normalized, so the pipeline feeds ``z * image_width``.
    z_values:
        Optional (N,) sequence of z values in pixel scale, used when the input
        landmarks are (N, 2).

    Returns
    -------
    (N, 2) or (N, 3) float ndarray, or None when the face scale is degenerate
    (IOD == 0, e.g. a pathological detection) — callers should record an error.
    """
    arr = np.asarray(landmarks, dtype=float)
    if arr.ndim != 2 or arr.shape[0] == 0 or arr.shape[1] < 2:
        raise ValueError("landmarks must be an (N, 2) or (N, 3) array-like with N > 0")

    xy = arr[:, :2]
    origin = xy[list(ALL_EYE_IDXS)].mean(axis=0)                       # mid-eyes point
    eye_right_center = xy[list(EYE_RIGHT_IDXS)].mean(axis=0)
    eye_left_center = xy[list(EYE_LEFT_IDXS)].mean(axis=0)
    scale = float(np.linalg.norm(eye_left_center - eye_right_center))  # IOD
    if scale <= 0:
        LOGGER.warning("Degenerate face scale (IOD=0); landmarks cannot be normalized.")
        return None

    normalized = (xy - origin) / scale
    if return_z:
        if arr.shape[1] >= 3:
            z = arr[:, 2]
        elif z_values is not None:
            z = np.asarray(z_values, dtype=float)
        else:
            raise ValueError(
                "return_z=True requires (N, 3) landmarks or explicit z_values "
                "(in the same pixel scale as x/y)"
            )
        if z.shape[0] != arr.shape[0]:
            raise ValueError("z_values length does not match the number of landmarks")
        normalized = np.column_stack([normalized, (z - z.mean()) / scale])
    return normalized


## 5. Ingestion-time validation

Bad inputs are **gated, logged, and reproducible** — never silently dropped (D10):
accepted images flow into the feature tables; failures land in `rejected_images.csv`
with machine-readable reasons. Thresholds live in `CONFIG` (D6) and are per-dataset
tunable — e.g. KDEF studio shots can afford a higher blur bar than webcam selfies.


In [ ]:
def validate_image(image_bgr: np.ndarray) -> Tuple[bool, List[str], Dict[str, float]]:
    """Check one BGR image for quality problems at ingestion time.

    Checks (thresholds in CONFIG, rationale in D6):
        * blur         — variance of the Laplacian (focus measure) below threshold
        * darkness     — mean grayscale below `brightness_low`
        * over-exposure— mean grayscale above `brightness_high`
        * tiny size    — short side below `min_resolution` px

    Returns
    -------
    (ok, issues, metrics):
        ok      : True when no issue fired
        issues  : list of human-readable reason strings (empty when ok)
        metrics : {"blur_score", "brightness", "width", "height"} — always computed,
                  so even rejected images keep quantitative provenance in the manifest.
    """
    issues: List[str] = []
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    blur_score = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    brightness = float(gray.mean())
    height, width = gray.shape[:2]

    if blur_score < CONFIG["blur_threshold"]:
        issues.append(f"blurry: laplacian_var={blur_score:.1f} < {CONFIG['blur_threshold']:g}")
    if brightness < CONFIG["brightness_low"]:
        issues.append(f"too dark: mean_gray={brightness:.1f} < {CONFIG['brightness_low']:g}")
    if brightness > CONFIG["brightness_high"]:
        issues.append(f"too bright: mean_gray={brightness:.1f} > {CONFIG['brightness_high']:g}")
    if min(width, height) < CONFIG["min_resolution"]:
        issues.append(
            f"low resolution: {width}x{height}, short side < {CONFIG['min_resolution']}px"
        )

    metrics = {
        "blur_score": blur_score,
        "brightness": brightness,
        "width": int(width),
        "height": int(height),
    }
    return (len(issues) == 0), issues, metrics


## 6. Face crop / alignment

Two outputs per face, both opt-in via `save_crops=True` (they're for **downstream CNN
experiments**, not required by the classical-ML tables):

* `crop_face` — landmark bounding box + `margin` (fraction of box size, default 0.20),
  clipped to image bounds. Roughly preserves context (hairline/chin).
* `align_face` — **similarity transform** (rotate + uniform scale, no shear) mapping the
  two eye centers to canonical positions `(0.35·W, 0.40·H)` and `(0.65·W, 0.40·H)` of a
  224×224 canvas (D7). Because the transform normalizes scale and roll, every aligned
  crop is directly comparable — useful later for CNN baselines.

The classical-ML tables do **not** depend on either: they are already IOD-normalized.


In [ ]:
def crop_face(
    image_bgr: np.ndarray,
    landmarks_px: Sequence[Sequence[float]],
    margin: Optional[float] = None,
) -> Optional[np.ndarray]:
    """Crop the face region (landmark bounding box + relative margin) from the image.

    Returns the cropped BGR array, or None when the box degenerates (< 2 px).
    """
    margin = CONFIG["crop_margin"] if margin is None else margin
    pts = np.asarray(landmarks_px, dtype=float)[:, :2]
    height, width = image_bgr.shape[:2]

    x_min, y_min = pts.min(axis=0)
    x_max, y_max = pts.max(axis=0)
    box_w, box_h = x_max - x_min, y_max - y_min

    x0 = int(max(0, math.floor(x_min - margin * box_w)))
    y0 = int(max(0, math.floor(y_min - margin * box_h)))
    x1 = int(min(width, math.ceil(x_max + margin * box_w)))
    y1 = int(min(height, math.ceil(y_max + margin * box_h)))
    if x1 - x0 < 2 or y1 - y0 < 2:
        LOGGER.warning("Degenerate crop box (%dx%d) — skipping crop.", x1 - x0, y1 - y0)
        return None
    return image_bgr[y0:y1, x0:x1].copy()


def align_face(
    image_bgr: np.ndarray,
    landmarks_px: Sequence[Sequence[float]],
    output_size: Optional[Tuple[int, int]] = None,
    eye_target_x: Tuple[float, float] = (0.35, 0.65),
    eye_target_y: float = 0.40,
) -> Optional[np.ndarray]:
    """Affine-align a face so the eye line is horizontal at canonical positions.

    Builds a similarity transform (rotation + uniform scale + translation) that maps
    the two eye centers to `(eye_target_x[i]·W, eye_target_y·H)` on an
    `output_size` canvas, then samples the image with that transform.

    Returns the aligned BGR image, or None on degenerate eye geometry.
    """
    output_size = output_size or CONFIG["align_output_size"]
    out_w, out_h = output_size
    pts = np.asarray(landmarks_px, dtype=float)[:, :2]

    src_right = pts[list(EYE_RIGHT_IDXS)].mean(axis=0)  # appears image-left (D9)
    src_left = pts[list(EYE_LEFT_IDXS)].mean(axis=0)    # appears image-right
    dst_right = np.array([eye_target_x[0] * out_w, eye_target_y * out_h])
    dst_left = np.array([eye_target_x[1] * out_w, eye_target_y * out_h])

    src_vec = src_left - src_right
    src_len = float(np.linalg.norm(src_vec))
    if src_len <= 0:
        LOGGER.warning("Degenerate eye geometry (zero-length eye line); cannot align.")
        return None
    scale = float(np.linalg.norm(dst_left - dst_right)) / src_len

    angle = math.atan2(src_vec[1], src_vec[0])           # current eye-line angle
    cos_a, sin_a = math.cos(-angle), math.sin(-angle)    # rotate eye line onto +x axis
    rotation = scale * np.array([[cos_a, -sin_a], [sin_a, cos_a]])

    src_mid = (src_right + src_left) / 2.0
    dst_mid = (dst_right + dst_left) / 2.0
    translation = dst_mid - rotation @ src_mid
    matrix = np.column_stack([rotation, translation])    # 2x3 affine

    return cv2.warpAffine(
        image_bgr, matrix, (out_w, out_h),
        flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REFLECT,
    )


## 7. Subject / source / label parsing

Implements the folder convention documented at the top of the notebook.
`parse_source` = first folder under `root`; `parse_subject_id` = deepest
non-label folder (filename fallback, with a generic-token blacklist so
`img_014.jpg` doesn't become subject "img"); `parse_label` = three-class
label from any folder component or filename token, else `None`.

**Generalization note:** the old repo's labeling was a binary red/green dot burned
into frame (20, 20) and read back per frame (`dados.cor`) — unusable for a 3-class
image dataset. Here labels come from dataset structure (folders/filenames), which is
how public datasets are shipped, and `None` means "annotate me in Scrub".


In [ ]:
def _relative_parts(path: PathLike, root: Optional[PathLike]) -> Tuple[str, ...]:
    """Path parts relative to ``root`` when possible, else the raw parts."""
    p = Path(path)
    if root is None:
        return tuple(p.parts)
    try:
        return tuple(p.relative_to(Path(root)).parts)
    except ValueError:
        try:
            return tuple(p.resolve().relative_to(Path(root).resolve()).parts)
        except (ValueError, OSError):
            return tuple(p.parts)


def _subject_from_filename(filename: str) -> Optional[str]:
    """Best-effort subject from a filename: token before the first separator.

    Returns None for generic tokens ("img", "photo", …) and pure numbers —
    those would create meaningless pseudo-subjects.
    """
    stem = Path(filename).stem
    token = re.split(r"[ _\-.]+", stem, maxsplit=1)[0] if stem else ""
    if not token or token.lower() in GENERIC_SUBJECT_TOKENS or token.isdigit():
        return None
    return token


def parse_source(path: PathLike, root: Optional[PathLike] = None) -> Optional[str]:
    """Dataset source = first directory level under ``root`` (see folder convention).

    ``<root>/kdef/smile/kevin_01.jpg`` -> ``"kdef"``. None when the image sits
    directly in ``root``.
    """
    parts = _relative_parts(path, CONFIG["image_dir"] if root is None else root)
    dirs = parts[:-1]
    return dirs[0] if dirs else None


def parse_subject_id(path: PathLike, root: Optional[PathLike] = None) -> Optional[str]:
    """Subject id = deepest non-label folder, else a filename token (see convention).

    Handles both ``<source>/<subject>/<label>/img.jpg`` and
    ``<source>/<label>/<subject>/img.jpg``; falls back to the first filename
    token when no subject folder exists.
    """
    parts = _relative_parts(path, CONFIG["image_dir"] if root is None else root)
    dirs = parts[:-1]
    non_label_dirs = [d for d in dirs if d.lower() not in CLASS_ALIASES]
    if len(non_label_dirs) >= 2:
        return non_label_dirs[-1]
    if len(non_label_dirs) == 1 and len(dirs) >= 2:
        return _subject_from_filename(parts[-1])  # only source + label folders present
    return _subject_from_filename(parts[-1])


def parse_label(path: PathLike, root: Optional[PathLike] = None) -> Optional[str]:
    """Three-class label from folder components or filename tokens, else None.

    Matches ``CLASS_ALIASES`` case-insensitively (includes PT aliases like
    ``neutro``/``triste``/``sorriso`` since the team's existing data uses them).
    """
    parts = _relative_parts(path, CONFIG["image_dir"] if root is None else root)
    for part in parts[:-1]:
        label = CLASS_ALIASES.get(part.lower())
        if label:
            return label
    tokens = re.split(r"[ _\-.]+", Path(parts[-1]).stem.lower())
    for token in tokens:
        label = CLASS_ALIASES.get(token)
        if label:
            return label
    return None


def parse_metadata(path: PathLike, root: Optional[PathLike] = None) -> Dict[str, Optional[str]]:
    """Convenience bundle: image_path, source, subject_id, label for one path."""
    return {
        "image_path": str(path),
        "source": parse_source(path, root),
        "subject_id": parse_subject_id(path, root),
        "label": parse_label(path, root),
    }


## 8. Single-pass processing core + manifest

Every table (geometric, landmark, manifest) consumes the **same** per-image
record generator — one validation pass, one FaceMesh pass per image, no duplicated
work when `run_obtain_pipeline` assembles all outputs (and no shared mutable state,
which is what corrupted the old repo's `tabelasBrutas` when looping over videos).

`build_manifest` writes `manifest.csv`: the dataset's provenance record — one row
per image found, accepted or not.


In [ ]:
def _safe_crop_stem(path: PathLike, root: PathLike) -> str:
    """Collision-free filename stem for crops: flattened relative path."""
    p = Path(path)
    try:
        rel = p.relative_to(Path(root))
    except ValueError:
        rel = p
    flat = "_".join(rel.with_suffix("").parts)
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", flat)


def _save_crops(image_bgr: np.ndarray, landmarks_px: Sequence[Sequence[float]],
                crops_dir: Path, path: PathLike, root: PathLike) -> None:
    """Write raw + aligned crops under ``crops_dir/raw`` and ``crops_dir/aligned``."""
    raw_dir, aligned_dir = crops_dir / "raw", crops_dir / "aligned"
    raw_dir.mkdir(parents=True, exist_ok=True)
    aligned_dir.mkdir(parents=True, exist_ok=True)
    stem = _safe_crop_stem(path, root)
    crop = crop_face(image_bgr, landmarks_px)
    if crop is not None:
        cv2.imwrite(str(raw_dir / f"{stem}.jpg"), crop)
    aligned = align_face(image_bgr, landmarks_px)
    if aligned is not None:
        cv2.imwrite(str(aligned_dir / f"{stem}.jpg"), aligned)


def _process_image(
    path: PathLike,
    image_bgr: Optional[np.ndarray],
    face_mesh: "mp.solutions.face_mesh.FaceMesh",
    root: PathLike,
    crops_dir: Optional[Path] = None,
    include_z: bool = False,
) -> Dict[str, Any]:
    """Process one image into a full record (metadata + validation + landmarks + features).

    This is the single-pass core shared by every table builder. Underscore-prefixed
    keys are heavy intermediates (landmark arrays) stripped before writing CSVs.
    """
    record: Dict[str, Any] = {
        "image_path": str(path),
        "source": parse_source(path, root),
        "subject_id": parse_subject_id(path, root),
        "label": parse_label(path, root),
        "width": np.nan,
        "height": np.nan,
        "blur_score": np.nan,
        "brightness": np.nan,
        "face_found": False,
        "n_landmarks": 0,
        "n_faces": 0,
        "rejected_reason": "",
        "error": None,
        "_landmarks_unit": None,
        "geometry": None,
    }
    if image_bgr is None:  # unreadable (load_images already logged it)
        record["rejected_reason"] = "unreadable image"
        record["error"] = "unreadable image"
        return record

    ok, issues, metrics = validate_image(image_bgr)
    record.update(metrics)
    if not ok:
        record["rejected_reason"] = "; ".join(issues)
        record["error"] = f"rejected at ingestion: {record['rejected_reason']}"
        return record

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    landmarks = extract_landmarks(image_rgb, face_mesh)
    record["n_faces"] = landmarks["n_faces"]
    record["error"] = landmarks["error"]
    if not landmarks["face_found"]:
        return record

    record["face_found"] = True
    record["n_landmarks"] = len(landmarks["landmarks_px"])
    try:
        record["geometry"] = compute_geometric_features(landmarks["landmarks_px"])
        # z handling (D1): FaceMesh's native z is roughly x-normalized, so scale it
        # to pixel units (z * width) before normalization gives it IOD units too.
        z_pixel = (
            [z * record["width"] for _, _, z in landmarks["landmarks_norm"]]
            if include_z
            else None
        )
        record["_landmarks_unit"] = normalize_landmarks(
            landmarks["landmarks_px"], return_z=include_z, z_values=z_pixel
        )
    except (ValueError, IndexError) as exc:
        record["error"] = f"feature extraction failed: {exc}"
        LOGGER.warning("%s: %s", path, exc)
        return record
    if record["_landmarks_unit"] is None:
        record["error"] = "landmark normalization failed (degenerate face scale)"

    if crops_dir is not None:
        _save_crops(image_bgr, landmarks["landmarks_px"], crops_dir, path, root)
    return record


def iter_processed_images(
    image_dir: PathLike,
    face_mesh: Optional["mp.solutions.face_mesh.FaceMesh"] = None,
    root: Optional[PathLike] = None,
    crops_dir: Optional[Path] = None,
    include_z: bool = False,
    extensions: Sequence[str] = CONFIG["extensions"],
) -> Generator[Dict[str, Any], None, None]:
    """Yield one processed record per image found under ``image_dir``.

    Creates (and closes) its own FaceMesh unless one is supplied. Individual
    failures are logged and yielded as rejected records — a single bad image
    never aborts the run.
    """
    parse_root = Path(image_dir if root is None else root)
    with face_mesh_session(face_mesh) as fm:
        for path, image in tqdm(load_images(image_dir, extensions), desc="Processing images"):
            try:
                yield _process_image(
                    path, image, fm, root=parse_root,
                    crops_dir=crops_dir, include_z=include_z,
                )
            except Exception as exc:  # absolute last-resort guard: keep the run alive
                LOGGER.error("Unexpected failure processing %s: %s", path, exc)
                record = parse_metadata(path, parse_root)
                record.update({
                    "width": np.nan, "height": np.nan, "blur_score": np.nan,
                    "brightness": np.nan, "face_found": False, "n_landmarks": 0,
                    "n_faces": 0, "rejected_reason": f"processing error: {exc}",
                    "error": f"processing error: {exc}",
                    "_landmarks_unit": None, "geometry": None,
                })
                yield record


def build_manifest(
    image_dir: PathLike,
    face_mesh: Optional["mp.solutions.face_mesh.FaceMesh"] = None,
    output_csv: Optional[PathLike] = None,
    root: Optional[PathLike] = None,
) -> pd.DataFrame:
    """Build and save ``manifest.csv`` — one provenance row per image found.

    Columns: image_path, source, subject_id, label, width, height, face_found,
    n_landmarks, blur_score, brightness, rejected_reason.
    Returns the manifest DataFrame.
    """
    output_csv = Path(CONFIG["output_dir"]) / "manifest.csv" if output_csv is None else Path(output_csv)
    rows = [
        {col: record.get(col) for col in MANIFEST_COLUMNS}
        for record in iter_processed_images(image_dir, face_mesh=face_mesh, root=root)
    ]
    manifest = pd.DataFrame(rows, columns=MANIFEST_COLUMNS)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    manifest.to_csv(output_csv, index=False)  # overwrite => idempotent (D8)
    LOGGER.info("Wrote %s (%d rows)", output_csv, len(manifest))
    return manifest


## 9. Geometric + landmark tables

`build_geometric_table(image_dir)` → `geometric_features.csv` (Approach A).
`build_landmark_table(image_dir)` → `landmark_features.csv` + `landmark_index.csv` (Approach B).

Both **skip validation-rejected images** (they live in `rejected_images.csv`) but keep
rows for accepted images where **no face was detected** — `face_found=False`, features
NaN, `error="no face detected"` — so face-detection failure stays visible in the tables
that feed Model stage, not just in the manifest.


In [ ]:
def build_geometric_table(
    image_dir: PathLike,
    face_mesh: Optional["mp.solutions.face_mesh.FaceMesh"] = None,
    output_csv: Optional[PathLike] = None,
    root: Optional[PathLike] = None,
) -> pd.DataFrame:
    """Build and save ``geometric_features.csv`` (Approach A — scalar geometry).

    One row per accepted image: metadata + face_found + IOD-normalized features
    (see ``compute_geometric_features``) + ``iod_px`` + ``error``.
    Returns the DataFrame.
    """
    output_csv = (
        Path(CONFIG["output_dir"]) / "geometric_features.csv"
        if output_csv is None else Path(output_csv)
    )
    rows: List[Dict[str, Any]] = []
    for record in iter_processed_images(image_dir, face_mesh=face_mesh, root=root):
        if record["rejected_reason"]:
            continue  # gated: rejected images are logged in rejected_images.csv (D10)
        row: Dict[str, Any] = {col: record.get(col) for col in GEOMETRIC_META_COLUMNS}
        geometry = record.get("geometry") or {}
        row["iod_px"] = geometry.get("iod_px", np.nan)
        for feature in GEOMETRIC_FEATURE_ORDER:
            row[feature] = geometry.get(feature, np.nan)
        row["error"] = record.get("error")
        rows.append(row)

    table = pd.DataFrame(rows, columns=GEOMETRIC_COLUMNS)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(output_csv, index=False)
    LOGGER.info("Wrote %s (%d rows)", output_csv, len(table))
    return table


def _landmark_index_frame(landmark_indices: Sequence[int], include_z: bool) -> pd.DataFrame:
    """Long-format mapping: column_name | face_mesh_index | axis."""
    rows: List[Dict[str, Any]] = []
    for index in landmark_indices:
        rows.append({"column_name": f"x_{index}", "face_mesh_index": index, "axis": "x"})
        rows.append({"column_name": f"y_{index}", "face_mesh_index": index, "axis": "y"})
        if include_z:
            rows.append({"column_name": f"z_{index}", "face_mesh_index": index, "axis": "z"})
    return pd.DataFrame(rows, columns=["column_name", "face_mesh_index", "axis"])


def build_landmark_table(
    image_dir: PathLike,
    face_mesh: Optional["mp.solutions.face_mesh.FaceMesh"] = None,
    include_z: Optional[bool] = None,
    landmark_indices: Optional[Sequence[int]] = None,
    output_csv: Optional[PathLike] = None,
    index_csv: Optional[PathLike] = None,
    root: Optional[PathLike] = None,
) -> pd.DataFrame:
    """Build and save ``landmark_features.csv`` + ``landmark_index.csv`` (Approach B).

    Wide format: one row per accepted image with flattened IOD-normalized
    coordinates ``x_i / y_i`` (and ``z_i`` when ``include_z``). Faces with no
    detection keep their metadata row with NaN coordinates.

    ``landmark_indices`` (default CONFIG["landmark_indices"] = all) selects a
    subset of FaceMesh indices; the companion index CSV always reflects exactly
    the columns written. Returns the feature DataFrame.
    """
    include_z = CONFIG["include_z"] if include_z is None else include_z
    landmark_indices = (
        CONFIG["landmark_indices"] if landmark_indices is None else landmark_indices
    )
    output_csv = (
        Path(CONFIG["output_dir"]) / "landmark_features.csv"
        if output_csv is None else Path(output_csv)
    )
    index_csv = (
        Path(CONFIG["output_dir"]) / "landmark_index.csv"
        if index_csv is None else Path(index_csv)
    )

    expected_count = 478 if CONFIG["refine_landmarks"] else 468
    indices_used = (
        list(range(expected_count)) if landmark_indices is None else list(landmark_indices)
    )

    rows: List[Dict[str, Any]] = []
    for record in iter_processed_images(
        image_dir, face_mesh=face_mesh, root=root, include_z=include_z
    ):
        if record["rejected_reason"]:
            continue  # gated (D10)
        row: Dict[str, Any] = {col: record.get(col) for col in LANDMARK_META_COLUMNS}
        unit = record.get("_landmarks_unit")
        if unit is not None:
            for index in indices_used:
                if index < unit.shape[0]:
                    row[f"x_{index}"] = float(unit[index, 0])
                    row[f"y_{index}"] = float(unit[index, 1])
                    if include_z and unit.shape[1] > 2:
                        row[f"z_{index}"] = float(unit[index, 2])
        rows.append(row)

    table = pd.DataFrame(rows)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(output_csv, index=False)
    _landmark_index_frame(indices_used, include_z).to_csv(index_csv, index=False)
    LOGGER.info("Wrote %s (%d rows, %d coordinate columns)", output_csv, len(table), len(indices_used) * (3 if include_z else 2))
    LOGGER.info("Wrote %s", index_csv)
    return table


## 10. Dataset-level summary (audit only — no splits, no modeling)

`summarize_dataset` answers the questions you need **before** Scrub:
how much data did we get, from whom, from which source; how often did face
detection succeed; why were images rejected; and — when labels exist in the
folder structure — how balanced are the three classes.


In [ ]:
def summarize_dataset(
    manifest: pd.DataFrame,
    geometric_df: Optional[pd.DataFrame] = None,
    landmark_df: Optional[pd.DataFrame] = None,
) -> Dict[str, Any]:
    """Print and return a dataset-level audit of the Obtain outputs.

    Aggregates (no splits, no modeling):
        * image counts per source and per subject
        * face-detection success rate over accepted images
        * rejection reasons (value counts)
        * label distribution (only for labels that exist in folder/file names)
    Cross-checks row counts of the feature tables against the manifest.
    """
    manifest = manifest.copy()
    manifest["rejected_reason"] = manifest["rejected_reason"].fillna("")
    rejected_mask = manifest["rejected_reason"] != ""
    n_total = len(manifest)
    n_rejected = int(rejected_mask.sum())
    accepted = manifest.loc[~rejected_mask]
    face_rate = float(accepted["face_found"].mean()) if len(accepted) else float("nan")

    summary: Dict[str, Any] = {
        "n_images": n_total,
        "n_accepted": n_total - n_rejected,
        "n_rejected": n_rejected,
        "face_detection_rate": face_rate,
        "images_per_source": manifest["source"].value_counts(dropna=False).to_dict(),
        "images_per_subject": manifest["subject_id"].value_counts(dropna=False).to_dict(),
        "rejection_reasons": manifest.loc[rejected_mask, "rejected_reason"].value_counts().to_dict(),
        "label_distribution": manifest["label"].value_counts(dropna=False).to_dict(),
        "label_distribution_accepted": accepted["label"].value_counts(dropna=False).to_dict(),
    }

    print("=" * 62)
    print("OBTAIN STAGE - DATASET SUMMARY")
    print("=" * 62)
    print(f"Images found           : {summary['n_images']}")
    print(f"Accepted (validated)   : {summary['n_accepted']}")
    print(f"Rejected at ingestion  : {summary['n_rejected']}")
    print(f"Face detection rate    : {face_rate:.1%} of accepted"
          if len(accepted) else "Face detection rate    : n/a (no accepted images)")
    print("-" * 62)
    print("Images per source:")
    for source, count in summary["images_per_source"].items():
        print(f"  {source!s:<30} {count}")
    print("Images per subject:")
    for subject, count in summary["images_per_subject"].items():
        print(f"  {subject!s:<30} {count}")
    print("Rejection reasons:")
    for reason, count in summary["rejection_reasons"].items():
        print(f"  {count:>5} x {reason}")
    print("Label distribution (accepted images):")
    for label, count in summary["label_distribution_accepted"].items():
        print(f"  {label!s:<30} {count}")
    print("-" * 62)
    if geometric_df is not None:
        expected = summary["n_accepted"]
        if len(geometric_df) != expected:
            print(f"WARNING: geometric_features rows = {len(geometric_df)}, expected {expected}")
        else:
            print(f"geometric_features rows = {len(geometric_df)} (ok)")
    if landmark_df is not None:
        expected = summary["n_accepted"]
        if len(landmark_df) != expected:
            print(f"WARNING: landmark_features rows = {len(landmark_df)}, expected {expected}")
        else:
            print(f"landmark_features rows  = {len(landmark_df)} (ok)")
    print("=" * 62)
    return summary


## 11. Pipeline runner (idempotent)

`run_obtain_pipeline(image_dir, output_dir)` = one pass over the dataset →
**all** CSVs + optional crops. Re-runs overwrite outputs and (when crops are
enabled) wipe `crops/` first, so row counts never grow across runs (D8) —
the direct fix for the old repo's accumulate-across-videos bug.


In [ ]:
def run_obtain_pipeline(
    image_dir: Optional[PathLike] = None,
    output_dir: Optional[PathLike] = None,
    save_crops: Optional[bool] = None,
    include_z: Optional[bool] = None,
    face_mesh: Optional["mp.solutions.face_mesh.FaceMesh"] = None,
    extensions: Sequence[str] = CONFIG["extensions"],
) -> Dict[str, pd.DataFrame]:
    """Run the full Obtain stage on a directory of images and write all outputs.

    Parameters
    ----------
    image_dir : root folder of images (default CONFIG["image_dir"] = ./data/images).
                Must exist — a missing directory raises FileNotFoundError.
    output_dir : destination for CSVs (default CONFIG["output_dir"] = ./outputs), created if needed.
    save_crops : write face crops to ``output_dir/crops/{raw,aligned}`` (default CONFIG["save_crops"]).
                 When True, ``crops/`` is wiped first — idempotent, no stale files.
    include_z : add z columns to the landmark table (default CONFIG["include_z"]).
    face_mesh : optionally reuse an existing FaceMesh instance (advanced).

    Returns
    -------
    dict of DataFrames: {"manifest", "rejected", "geometric", "landmarks", "landmark_index"}.
    """
    image_dir = Path(image_dir or CONFIG["image_dir"])
    output_dir = Path(output_dir or CONFIG["output_dir"])
    if not image_dir.is_dir():
        raise FileNotFoundError(f"Input image directory not found: {image_dir.resolve()}")
    output_dir.mkdir(parents=True, exist_ok=True)

    save_crops = CONFIG["save_crops"] if save_crops is None else save_crops
    include_z = CONFIG["include_z"] if include_z is None else include_z

    crops_dir = output_dir / "crops"
    if save_crops:
        if crops_dir.exists():
            shutil.rmtree(crops_dir)  # idempotency: no stale crops across runs (D8)
        crops_dir.mkdir(parents=True, exist_ok=True)

    records = list(
        iter_processed_images(
            image_dir, face_mesh=face_mesh, root=image_dir,
            crops_dir=crops_dir if save_crops else None,
            include_z=include_z, extensions=extensions,
        )
    )
    if not records:
        LOGGER.warning("No images found under %s — empty tables will be written.", image_dir)

    # --- manifest + rejection log -------------------------------------------
    manifest = pd.DataFrame(
        [{col: record.get(col) for col in MANIFEST_COLUMNS} for record in records],
        columns=MANIFEST_COLUMNS,
    )
    rejected_mask = manifest["rejected_reason"].fillna("") != ""
    rejected = manifest.loc[rejected_mask, REJECTED_COLUMNS].copy()

    # --- geometric table (Approach A) ----------------------------------------
    geom_rows: List[Dict[str, Any]] = []
    for record in records:
        if record["rejected_reason"]:
            continue
        row = {col: record.get(col) for col in GEOMETRIC_META_COLUMNS}
        geometry = record.get("geometry") or {}
        row["iod_px"] = geometry.get("iod_px", np.nan)
        for feature in GEOMETRIC_FEATURE_ORDER:
            row[feature] = geometry.get(feature, np.nan)
        row["error"] = record.get("error")
        geom_rows.append(row)
    geometric = pd.DataFrame(geom_rows, columns=GEOMETRIC_COLUMNS)

    # --- landmark table (Approach B) + index ---------------------------------
    expected_count = 478 if CONFIG["refine_landmarks"] else 468
    indices_used = list(range(expected_count)) if CONFIG["landmark_indices"] is None else list(CONFIG["landmark_indices"])
    lm_rows: List[Dict[str, Any]] = []
    for record in records:
        if record["rejected_reason"]:
            continue
        row = {col: record.get(col) for col in LANDMARK_META_COLUMNS}
        unit = record.get("_landmarks_unit")
        if unit is not None:
            for index in indices_used:
                if index < unit.shape[0]:
                    row[f"x_{index}"] = float(unit[index, 0])
                    row[f"y_{index}"] = float(unit[index, 1])
                    if include_z and unit.shape[1] > 2:
                        row[f"z_{index}"] = float(unit[index, 2])
        lm_rows.append(row)
    landmarks_df = pd.DataFrame(lm_rows)
    landmark_index = _landmark_index_frame(indices_used, include_z)

    # --- write everything (overwrite => idempotent) ---------------------------
    manifest.to_csv(output_dir / "manifest.csv", index=False)
    rejected.to_csv(output_dir / "rejected_images.csv", index=False)
    geometric.to_csv(output_dir / "geometric_features.csv", index=False)
    landmarks_df.to_csv(output_dir / "landmark_features.csv", index=False)
    landmark_index.to_csv(output_dir / "landmark_index.csv", index=False)
    LOGGER.info(
        "Obtain complete: %d manifest rows, %d rejected, %d geometric rows, %d landmark rows",
        len(manifest), len(rejected), len(geometric), len(landmarks_df),
    )
    return {
        "manifest": manifest,
        "rejected": rejected,
        "geometric": geometric,
        "landmarks": landmarks_df,
        "landmark_index": landmark_index,
    }


## 12. Optional helper — mining stills from existing videos

Not used by the pipeline above (Obtain here is image-first), but handy when the
team wants extra frames from session videos such as the old repo's `videos/`.
Returns `(name, frame_bgr)` pairs with a synthetic `folder::frame_000123` name.


In [ ]:
def read_video_frames(
    video_path: PathLike,
    step: int = 1,
    max_frames: Optional[int] = None,
) -> Generator[Tuple[str, np.ndarray], None, None]:
    """OPTIONAL: yield ``(name, frame_bgr)`` pairs from a video file.

    Parameters
    ----------
    video_path : video file path.
    step       : take every `step`-th frame (1 = all frames).
    max_frames : stop after this many yielded frames (None = until the video ends).
    """
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        LOGGER.warning("Could not open video: %s", video_path)
        return
    yielded, frame_idx = 0, 0
    try:
        while True:
            success, frame = capture.read()
            if not success:
                break
            if frame_idx % step == 0:
                yield f"{Path(video_path).stem}::frame_{frame_idx:06d}", frame
                yielded += 1
                if max_frames is not None and yielded >= max_frames:
                    break
            frame_idx += 1
    finally:
        capture.release()


## 13. Demo

Create `./data/images` with your dataset (public or team photos, following the
folder convention), then uncomment:


In [ ]:
# ============================================================================
# DEMO — uncomment when your dataset is in place.
#
# results = run_obtain_pipeline(
#     image_dir="./data/images",
#     output_dir="./outputs",
#     save_crops=False,      # True also writes crops/raw/ and crops/aligned/
#     include_z=False,       # True adds z_* columns to landmark_features.csv
# )
#
# results["manifest"].head()
# results["geometric"].describe()
# summarize_dataset(results["manifest"], results["geometric"], results["landmarks"])
# ============================================================================


## 14. Outputs & schemas (quick reference)

**`manifest.csv`** — one row per image found (accepted or not):
`image_path, source, subject_id, label, width, height, face_found, n_landmarks, blur_score, brightness, rejected_reason`

**`rejected_images.csv`** — subset of the manifest with non-empty `rejected_reason`
(blurry / too dark / too bright / low resolution / unreadable / processing error).

**`geometric_features.csv`** — one row per *accepted* image:
`image_path, source, subject_id, label, face_found, n_faces, iod_px, ear_left, ear_right, mar, mouth_width, mouth_height, brow_eye_distance_left, brow_eye_distance_right, mouth_curvature, nose_to_mouth_corner_left, nose_to_mouth_corner_right, face_width, face_height, error`
— all distances in IOD units; `ear`/`mar` dimensionless; `mouth_curvature` negative = smile-like.

**`landmark_features.csv`** — one row per accepted image:
`image_path, source, subject_id, label, face_found, n_landmarks, error, x_0…x_477, y_0…y_477`
(Ith column = IOD-normalized coordinate of FaceMesh landmark *i*, centered on the mid-eyes point).

**`landmark_index.csv`** — `column_name, face_mesh_index, axis` — the long-format
map from every coordinate column back to its FaceMesh landmark index.

**`crops/raw/`, `crops/aligned/`** — optional 224×224-aligned + raw face crops
(only when `save_crops=True`).


## 15. Limitations & future Obtain work

* **Landmark failures under pose/lighting.** FaceMesh degrades for large yaw/pitch,
  occlusions and strong shadows; failures surface as `face_found=False` rows, not crashes.
  Audit `manifest.csv` before blaming the model stage.
* **No public-dataset downloader yet.** FER2013/CK+/AffectNet/KDEF/RAF-DB must be
  downloaded manually (licences vary); drop them under `./data/images/<dataset>/…` and
  the pipeline ingests them unchanged. A scripted downloader with checksums is future work.
* **No 3-class annotation tool.** Labels are only *harvested* from folder/file names;
  rows with `label=None` need annotation (a small labeling UI is future Obtain/Scrub work).
* **No train/val/test split and no augmentation** — deliberately: Obtain ends at a clean,
  audited, feature-complete dataset. Splitting belongs to Scrub/Model.
* **Validation thresholds are heuristics.** Laplacian-variance blur detection flags
  textured-but-sharp images and misses motion blur on textured backgrounds; tune per dataset.
* **mediapipe solutions API is legacy.** `mp.solutions` (pinned `mediapipe==0.10.21` here,
  same as the old repo) still works but is deprecated upstream; migrating to the
  MediaPipe Tasks API (face landmarker) will eventually be required.
* **Single-face assumption.** Group photos are handled by largest-face selection, but the
  ignored faces are not recorded; a multi-face extension would need one row per face.
